In [4]:
# ============================================
# TRAIN MODELS WITH FEATURE ENGINEERING (80/20)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import recall_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import joblib
import os

# -----------------------------
# Load real dataset
# -----------------------------
import os
data_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'Maternal Health Risk Data Set.csv')
df = pd.read_csv(data_path)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
df["MAP"] = (df["SystolicBP"] + 2 * df["DiastolicBP"]) / 3
df["PulsePressure"] = df["SystolicBP"] - df["DiastolicBP"]
df["ShockIndex"] = df["HeartRate"] / df["SystolicBP"]
df["BPRatio"] = df["SystolicBP"] / df["DiastolicBP"]

df["TempDeviation"] = df["BodyTemp"] - 98.2
df["BSDeviation"] = df["BS"] - 7

df["CombinedRiskScore"] = (
    (df["MAP"] > 105).astype(int) +
    (df["BS"] > 10).astype(int) +
    (df["HeartRate"] > 90).astype(int) +
    (df["TempDeviation"] > 1).astype(int)
)

# -----------------------------
# Split X and y
# -----------------------------
X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

# -----------------------------
# Encode labels
# -----------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# -----------------------------
# 80/20 Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# -----------------------------
# Define models
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000))
    ]),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_enc)),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    )
}

# Voting & Stacking Ensembles
estimators = [
    ("lr", LogisticRegression(max_iter=5000)),
    ("dt", DecisionTreeClassifier()),
    ("rf", RandomForestClassifier()),
    ("gb", GradientBoostingClassifier())
]

models["Voting Ensemble"] = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

models["Stacking Ensemble"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=5000)
)

# -----------------------------
# Train + Evaluate + Save Models
# -----------------------------
results = {}

os.makedirs("../models_FE", exist_ok=True)

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    macro_recall = recall_score(y_test, preds, average="macro")
    results[name] = macro_recall
    
    print(f"Macro Recall: {macro_recall:.4f}")
    print(classification_report(y_test, preds))
    
    # Save model
    joblib.dump(model, f"../models_FE/{name.replace(' ', '_')}.pkl")

# -----------------------------
# Comparison Table
# -----------------------------
comparison_df = pd.DataFrame.from_dict(results, orient="index", columns=["Macro Recall"])
comparison_df = comparison_df.sort_values(by="Macro Recall", ascending=False)

comparison_df



Training Logistic Regression...
Macro Recall: 0.6750
              precision    recall  f1-score   support

           0       0.88      0.80      0.84        55
           1       0.62      0.85      0.72        81
           2       0.60      0.37      0.46        67

    accuracy                           0.68       203
   macro avg       0.70      0.67      0.67       203
weighted avg       0.68      0.68      0.67       203


Training Decision Tree...
Macro Recall: 0.8352
              precision    recall  f1-score   support

           0       0.96      0.91      0.93        55
           1       0.91      0.72      0.80        81
           2       0.68      0.88      0.77        67

    accuracy                           0.82       203
   macro avg       0.85      0.84      0.83       203
weighted avg       0.85      0.82      0.83       203


Training Random Forest...
Macro Recall: 0.8712
              precision    recall  f1-score   support

           0       0.96      0.95

,Macro Recall
Random Forest,0.871198
Voting Ensemble,0.859937
XGBoost,0.849987
Decision Tree,0.835246
Stacking Ensemble,0.832256
Gradient Boosting,0.823390
Logistic Regression,0.674995


In [5]:
# ============================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

for name, model in models.items():
    # Wrap model in an imbalanced-learn Pipeline so SMOTE is applied
    # inside each fold only — preventing data leakage
    cv_pipeline = ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])
    
    scores = cross_val_score(
        cv_pipeline, X, y_enc,
        cv=cv,
        scoring="recall_macro",
        n_jobs=-1
    )
    
    cv_results[name] = {
        "Mean Macro Recall": scores.mean(),
        "Std": scores.std()
    }
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results).T
cv_df = cv_df.sort_values("Mean Macro Recall", ascending=False)
cv_df

Logistic Regression: 0.6315 ± 0.0199
Decision Tree: 0.8530 ± 0.0264
Random Forest: 0.8550 ± 0.0254
Gradient Boosting: 0.8012 ± 0.0177
XGBoost: 0.8579 ± 0.0208
Voting Ensemble: 0.8662 ± 0.0191
Stacking Ensemble: 0.8592 ± 0.0208


,Mean Macro Recall,Std
Voting Ensemble,0.866211,0.019073
Stacking Ensemble,0.859181,0.020833
XGBoost,0.857891,0.020789
Random Forest,0.855033,0.025450
Decision Tree,0.852990,0.026374
Gradient Boosting,0.801192,0.017722
Logistic Regression,0.631460,0.019867
